# Summary statistics of dataset

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm
import os
from pathlib import Path

In [ ]:
# Import processed data from csv
# Define the path to the processed data based on the current user
current_user = os.getlogin()
PROC_DIR = Path(rf"/home/{current_user}/local-share/processed")

# Read the processed data
data = pd.read_csv(PROC_DIR / "features_standardized.csv") 

# Total number of students and drop-outs

In [ ]:
# Count total number of students and drop-outs
total_students = data['SINH_ID'].nunique()
total_dropouts = data[data['sdo5_degree_drop_out'] == 1]['SINH_ID'].nunique()
drop_out_rate = total_dropouts / total_students * 100

# Print the results
print(f"Total number of students: {total_students}")
print(f"Total number of drop-outs: {total_dropouts}")
print(f"Drop-out rate: {drop_out_rate:.2f}%")

## Number of students and drop-outs per year

In [ ]:
# Visualise number of continuing freshman and the number of drop-outs of 'sdo5_degree_drop_out' per sdo5_degree_COLLEGEJAAR
sns.countplot(x='sdo5_degree_COLLEGEJAAR', hue='sdo5_degree_drop_out', data=data)
plt.title('Number of Continuing Freshman and Drop-outs per College Year')
plt.xlabel('College Year')
plt.ylabel('Count')
plt.legend(title='', labels=['Continued studying', 'Dropped Out'])
plt.show()

# Counts and percentages (drop-out rate) per year
for year in sorted(data['sdo5_degree_COLLEGEJAAR'].unique()):
    year_data = data[data['sdo5_degree_COLLEGEJAAR'] == year]
    total_students = len(year_data)
    dropouts = year_data['sdo5_degree_drop_out'].sum()
    dropout_rate = (dropouts / total_students) * 100
    print(f"Year: {year}, Total Students: {total_students}, Drop-outs: {dropouts}, Dropout Rate: {dropout_rate:.2f}%")

## Number of drop-outs during the college year

In [ ]:
# Visualise drop-out numbers throughout the college year
data['dropout_date'] = pd.to_datetime(data['dropout_date'])
data['dropout_month'] = data['dropout_date'].dt.month
dropout_counts = data[data['sdo5_degree_drop_out'] == 1].groupby('dropout_month').size()

# Add column with percentage of drop-outs per month
dropout_counts = dropout_counts.reset_index(name='dropout_count')
dropout_counts['dropout_percentage'] = (dropout_counts['dropout_count'] / dropout_counts['dropout_count'].sum()) * 100

# Add column with college year month numbers
dropout_counts['college_year_month'] = dropout_counts['dropout_month'] - 8
dropout_counts.loc[dropout_counts['college_year_month'] <= 0, 'college_year_month'] += 12

# Add month names for better readability
month_names = {1: 'September', 2: 'October', 3: 'November', 4: 'December', 5: 'January', 6: 'February',
               7: 'March', 8: 'April', 9: 'May', 10: 'June', 11: 'July', 12: 'August'}
dropout_counts['month_name'] = dropout_counts['dropout_month'].map(month_names)

print(dropout_counts)

# Plot drop-out counts per month
sns.barplot(x='college_year_month', y='dropout_count', data=dropout_counts)
plt.title('Number of Drop-outs per Month')
plt.xlabel('Collegeyear month (numbered from September)')
plt.ylabel('Number of Drop-outs')
plt.show()

## Drop-out per degree

In [ ]:
# Look at the drop-out rate per degree (sdo5_degree_degree)
degree_dropout = data.groupby('sdo5_degree_degree')['sdo5_degree_drop_out'].agg(['sum', 'count'])
degree_dropout['dropout_rate'] = degree_dropout['sum'] / degree_dropout['count']
degree_dropout['dropout_rate_prc'] = degree_dropout['sum'] / degree_dropout['count'] * 100

# Visualize drop-out rate per degree in a vertical bar plot
degree_dropout = degree_dropout.sort_values(by='dropout_rate_prc', ascending=False) # Order data by drop-out rate
plt.figure(figsize=(12, 6))
sns.barplot(x=degree_dropout.index, y='dropout_rate_prc', data=degree_dropout.reset_index())
plt.title('Drop-out rate per HU degree')
plt.xlabel('Degree')
plt.ylabel('Drop-out (%)')
plt.xticks(rotation=90, ha='right')
plt.ylim(0, 100)
plt.axhline(y=drop_out_rate, color='r', linestyle='--', label='Average drop-out (%)')
plt.legend()    
plt.show()

## Types of drop-out categories
Please note that drop-out categories are not mutually exclusive: e.g. a student who drops out without a propedeuse can also switch to another degree within the HU. 

In [ ]:
# Import cleaned but unprocessed data from csv
data = pd.read_csv(PROC_DIR / "v2_combined.csv") 

In [ ]:
# Count students in each drop-out category:
# - temporary drop-out (sdo5_degree_drop_out_temporary)
# - switching program within HU (sdo5_degree_drop_out_to_other_degree_in_HU)
# - drop-out with propedeuse (sdo5_degree_drop_out_with_propedeuse)
# - drop-out without propedeuse (sdo5_degree_drop_out_without_propedeuse)
dropout_categories = [
    'sdo5_degree_drop_out_temporary',
    'sdo5_degree_drop_out_to_other_degree_in_HU',
    'sdo5_degree_drop_out_with_propedeuse',
    'sdo5_degree_drop_out_without_propedeuse'
]

dropout_categories_labels = [
    'Temporary drop-out',
    'Switching within HU',
    'Drop-out with propedeuse',
    'Drop-out without propedeuse'
]

total_dropouts = data[data['sdo5_degree_drop_out'] == 1]['SINH_ID'].nunique()

# Visualize drop-out category counts in a table
dropout_counts = {category: data[category].sum() for category in dropout_categories}
dropout_df = pd.DataFrame(list(dropout_counts.items()), columns=['Drop-out category', 'Number of students'])
dropout_df['Drop-out category'] = dropout_categories_labels

# Add percentage column that computes the percentage of students in each drop-out category to the total number of drop-outs
dropout_df['Percentage'] = (dropout_df['Number of students'] / total_dropouts) * 100
print(dropout_df)

### Results
The above results show that students that drop-out with a propedeuse, or drop out temporarily (i.e. discontinue their studies for less than 1 year) are quite rare (<0.1% and < 2% of drop-outs respectively). This means that almost all students that drop-out do so without propedeuse (99%), and most students also do not switch to a different degree within the HU (79%). 